In [ ]:

!pip -q install -U langgraph

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. Define the shared state
class AgentState(TypedDict):
    query: str
    attempts: int
    result: str
    relevant: bool
    answer: str

# 2. Search node
def search(state: AgentState):
    attempt = state["attempts"] + 1
    print(f"\nSearching... Attempt {attempt}")

    # Simulated search results (no API key required)
    if attempt == 1:
        result = "An article about database management systems."
    else:
        result = "A research paper about transformer models for medical image classification."

    print("Result:", result)

    return {
        "attempts": attempt,
        "result": result
    }

# 3. Evaluate the search result
def evaluate(state: AgentState):
    relevant = "transformer" in state["result"].lower()
    print("Relevant:", relevant)

    return {"relevant": relevant}

# 4. Decide what happens next
def route(state: AgentState):
    if state["relevant"]:
        return "finish"
    elif state["attempts"] < 3:
        return "retry"
    else:
        return "finish"

# 5. Generate the final answer
def finish(state: AgentState):
    if state["relevant"]:
        answer = (
            f"Research found for '{state['query']}': "
            f"{state['result']}"
        )
    else:
        answer = "No relevant result found after 3 attempts."

    return {"answer": answer}

# 6. Build the LangGraph workflow
graph = StateGraph(AgentState)

graph.add_node("search", search)
graph.add_node("evaluate", evaluate)
graph.add_node("finish", finish)

graph.add_edge(START, "search")
graph.add_edge("search", "evaluate")

graph.add_conditional_edges(
    "evaluate",
    route,
    {
        "retry": "search",
        "finish": "finish"
    }
)

graph.add_edge("finish", END)

app = graph.compile()

# 7. Run the agent
initial_state = {
    "query": "Find research on transformer models",
    "attempts": 0,
    "result": "",
    "relevant": False,
    "answer": ""
}

output = app.invoke(initial_state)

print("\nFINAL ANSWER:")
print(output["answer"])
print("Total attempts:", output["attempts"])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.6/259.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 13.4 MB/s eta 0:00:00

Searching... Attempt 1
Result: An article about database management systems.
Relevant: False

Searching... Attempt 2
Result: A research paper about transformer models for medical image classification.
Relevant: True

FINAL ANSWER:
Research found for 'Find research on transformer models': A research paper about transformer models for medical image classification.
Total attempts: 2
